In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch


def git_output(*args):
    return subprocess.check_output(["git", *args], text=True).strip()


repo_from_git = Path(git_output("rev-parse", "--show-toplevel")).resolve()
print("Python:", sys.executable)
print("Working directory:", Path.cwd())
print("Git commit:", git_output("rev-parse", "HEAD"))
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())


# Environment and path audit

This notebook performs bounded, read-only checks. It does not load a model, traverse a dataset, modify assets, or start training.

In [ ]:
branch = git_output("branch", "--show-current")
commit = git_output("rev-parse", "HEAD")
worktree_status = git_output("status", "--short")

print("Git branch:", branch or "DETACHED")
print("Git commit:", commit)
print("Worktree clean:", not bool(worktree_status))
if worktree_status:
    print(worktree_status)


In [ ]:
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(
        f"GPU {index}: {properties.name}, "
        f"{properties.total_memory / 1024**3:.1f} GiB"
    )


In [ ]:
environment_names = (
    "DFDHR_REPO_ROOT",
    "DFDHR_PYTHON",
    "DFDHR_DATA_ROOT",
    "DFDHR_JSON_ROOT",
    "DFDHR_RUNTIME_ROOT",
    "DFDHR_ARCHIVE_ROOT",
    "DFDHR_CACHE_ROOT",
)

for name in environment_names:
    value = os.environ.get(name)
    print(f"{name}: {value if value else '<unset>'}")

required_names = (
    "DFDHR_REPO_ROOT",
    "DFDHR_PYTHON",
    "DFDHR_DATA_ROOT",
    "DFDHR_RUNTIME_ROOT",
)
missing = [name for name in required_names if not os.environ.get(name)]
assert not missing, f"Missing required environment variables: {missing}"
assert Path(os.environ["DFDHR_PYTHON"]).resolve() == Path(sys.executable).resolve()


In [ ]:
repo_root = Path(os.environ["DFDHR_REPO_ROOT"]).expanduser().resolve()
data_root = Path(os.environ["DFDHR_DATA_ROOT"]).expanduser().resolve()
runtime_root = Path(os.environ["DFDHR_RUNTIME_ROOT"]).expanduser().resolve()
json_root = Path(
    os.environ.get("DFDHR_JSON_ROOT", repo_root / "preprocessing/dataset_json")
).expanduser().resolve()

paths = {
    "repository": repo_root,
    "data": data_root,
    "JSON registry": json_root,
    "runtime": runtime_root,
}
for label, path in paths.items():
    print(f"{label}: exists={path.exists()} directory={path.is_dir()} path={path}")

assert repo_root == repo_from_git, "DFDHR_REPO_ROOT does not match the active Git repository"


In [ ]:
usage = shutil.disk_usage(runtime_root)
print(f"Runtime filesystem total: {usage.total / 1024**3:.1f} GiB")
print(f"Runtime filesystem free: {usage.free / 1024**3:.1f} GiB")
print(f"Runtime filesystem used: {usage.used / usage.total:.1%}")


In [ ]:
import yaml


def is_within(path, root):
    try:
        path.relative_to(root)
        return True
    except ValueError:
        return False


runtime_is_safe = not is_within(runtime_root, repo_root) and not is_within(
    runtime_root, data_root
)
print("Runtime root outside repository and data root:", runtime_is_safe)

config_results = []
config_dir = repo_root / "training/config/detector"
for config_path in sorted(config_dir.glob("*.yaml")):
    with config_path.open(encoding="utf-8") as handle:
        config = yaml.safe_load(handle) or {}
    for key in ("log_dir", "logdir"):
        value = config.get(key)
        if not isinstance(value, str) or not value:
            continue
        target = Path(value).expanduser()
        if not target.is_absolute():
            target = (repo_root / target).resolve()
        else:
            target = target.resolve()
        safe = not is_within(target, repo_root) and not is_within(target, data_root)
        config_results.append((config_path.name, key, value, safe))

for config_name, key, value, safe in config_results:
    print(f"{config_name} {key}={value!r}: outside protected roots={safe}")

print("All inspected config outputs outside protected roots:", all(
    result[-1] for result in config_results
))
assert runtime_is_safe, "DFDHR_RUNTIME_ROOT must be outside the repository and data root"
